In [1]:
# 필요한 라이브러리 설치
# 주의: 코랩 환경에서는 '!'를 붙여야 shell 명령어로 실행됩니다.

# transformers 라이브러리 설치 (모델 로드 및 추론에 사용)
!pip install transformers==4.42.3
!pip install accelerate==0.31.0
!pip install bitsandbytes==0.43.1
!pip install triton==2.3.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 MB 5.7 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.2.0
    Uninstalling triton-3.2.0:
      Successfully uninstalled triton-3.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires triton==3.2.0; platform_system == "Linux" and platform_machine == "x86_64", but you have triton 2.3.0 which is incompatible.


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# 모델 ID 정의
# Qwen1.5-4B-Chat 모델은 Qwen에서 공개한 한국어/중국어/영어 대응 대화형 모델입니다.
# Hugging Face에 등록되어 있어 모델 이름으로 바로 불러올 수 있습니다.
model_id = "Qwen/Qwen1.5-4B-Chat"

# 1. 4비트 양자화 설정
# BitsAndBytesConfig를 사용해 최신 방식으로 4bit 로드 설정을 지정합니다.
# - load_in_4bit=True: 4비트 양자화 사용
# - bnb_4bit_use_double_quant=True: 2중 양자화를 통해 메모리 사용량 절감
# - bnb_4bit_quant_type="nf4": 최신 NF4 양자화 방식 사용
# - bnb_4bit_compute_dtype=torch.float16: 연산은 FP16으로 수행
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 2. 토크나이저 로드
# 모델에 맞는 토크나이저를 자동으로 불러옵니다.
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 3. 모델 로드
# device_map="auto": Colab GPU에 자동 배치
# quantization_config=bnb_config: 위에서 정의한 4bit 양자화 설정 적용
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config
)

print("✅ 모델 로드 완료!")
# 모델 메모리 사용량 출력 (GB 단위)
print(f"모델의 메모리 사용량: {model.get_memory_footprint() / 1024**3:.2f} GB")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ 모델 로드 완료!
모델의 메모리 사용량: 3.55 GB


In [8]:
chat_history = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "LLM에 대해 5문장으로 요약해줘."}
]
# 2. 챗 템플릿 적용 및 토큰화
input_ids = tokenizer.apply_chat_template(
    chat_history,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)
# 3. 텍스트 생성 (추론)
output = model.generate(
    input_ids,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id
)
# 4. 생성된 결과를 텍스트로 변환 및 출력
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print("생성된 답변:\n")
print(generated_text)

생성된 답변:

system
You are a helpful assistant.
user
LLM에 대해 5문장으로 요약해줘.
assistant
LLM (Legal Studies)는 법학 전공을 포함하는 학위를 의미합니다. 이 학위를 취득하면 전문가로서 법적 분야에서 보안, 법률 및 법률 프로그램의 지식과 능력을 갖춘 자격을 얻을 수 있습니다.
LLM 학위에서는 다양한 분야의 법적 문제와 해결 방법을 다루며, 이를 바탕으로 전문적인 법률 전문가로 성장할 수 있는 데 도움이 됩니다. 또한,法学 연구 및 논술 기술을 배우면, 새로운 법률 제도나 법률 사례에 대한 적극적인 접근 방식을 개발할 수 있습니다.
LLM 학위를 취득하기 위해서는 대학이나 뉴스턴드러스 등에서 법학과 관련 교육을 받거나 법학 학습 과정을 통해 학습해야 합니다. 또한, 일주일 근무 시간과 인력 소비, 경제적 비용 등에 대한 고려도 중요합니다.
따라서, LL.M은 전문가로서 법적 분야에서 보안, 법률 및 법률 프로그램의 지식과
